In [2]:
import sys
root_path = '../' # This defaults to './', which should be the repository path
sys.path.append(root_path)
sys.path.append("./")

%load_ext autoreload
%autoreload 2

from utils.config import path_to_pypsa_network_sclopf, path_to_cascade_results
from scripts.run_cascade_code import run_cascade_dual_line_failures
from utils.plot_mitigation_strategies import (
    data_line_mitigation_diff_nnlines,
    plot_bar_histograms_diff_nnlines,
)
from utils import data_handling
import networkx as nx
from utils import cascade_simulation, extend_transmission_capacity
import pickle
import gzip
import pandas as pd
from datetime import datetime as dt
from tqdm import tqdm


In [10]:
save_all_cascades = False
check_n1_security = True
stop_timestamp_str = "2013-01-01 00:00"
n_nodes = 400
co2l = 0.1
use_sclopf = True
save_path = ""
snet_index = 0
nn_lines = 1
delta_num_parallel = 1
line_mitigation_dict = {
            "nn_links": nn_lines,
            "delta_num_parallel": delta_num_parallel,
            "extension_type": "most_likely_primary",
        }

path_to_pypsa_network_lopf = "./data/European_networks_lopf/"
save_path_lopf = "./results/lopf/cascade_results/"
save_path_sclopf = path_to_cascade_results


path_to_pypsa_network = path_to_pypsa_network_sclopf
full_path_to_file = (
    path_to_pypsa_network
    + f"sclopf-elec_s_{n_nodes}_ec_lv1.0_Co2L{co2l:.1f}-2920SEG.nc"
)

if use_sclopf:
    save_path = save_path_sclopf
else:
    save_path = save_path_lopf

fpath_out = save_path + f"system_splits_Co2L{co2l}_n{n_nodes}"

if not use_sclopf:
    fpath_out += "_lopf"

# Load PyPSA network, the graph of the subnetwork and its matrices
network = data_handling.load_pypsa_network(full_path_to_file, use_sclopf)
nx_graph = data_handling.build_networkx_graph(network, snet_index=snet_index)

# Check if line extension mitigation is supposed to be run.
# Networkx Graph is being modified, if line extension needs to be considered.
if line_mitigation_dict is None:
    I_m, B_d, num_parallels, line_limits = data_handling.get_matrices_from_nx_graph(
        nx_graph
    )

    # Calculate possible N-1 and N-2 failures (using non-bridges)
    bridge_idxs = data_handling.nx_edges_to_matrix_indices(
        nx.bridges(nx_graph), nx_graph
    )
    n_2_failures = cascade_simulation.calc_possible_double_line_failures(
        num_parallels, ignored_idxs=bridge_idxs, use_sclopf=use_sclopf
    )
    n_1_failures = cascade_simulation.calc_possible_single_line_failures(
        num_parallels, ignored_idxs=bridge_idxs
    )

else:
    # Cascade simulation with extended lines
    nn_links_extended = line_mitigation_dict["nn_links"]
    delta_num_parallel = line_mitigation_dict["delta_num_parallel"]
    extension_type = line_mitigation_dict["extension_type"]
    casc_dict = None

    if extension_type == "most_likely_primary":

        # Load previously generated cascade results
        with gzip.open(fpath_out + ".pklz", "rb") as fh_in:
            casc_dict = pickle.load(fh_in)

        # Check if all times have a cascade entry
        set_diff_len = len(
            set(network.snapshots) - set(pd.to_datetime(list(casc_dict.keys())))
        )
        assert (
            set_diff_len == 0
        ), f"Cascade dict does not have same entries but {set_diff_len} different ones!"

        print(
            "\n## Running for line extension: Loading previously found cascade dictionary!\n"
            + " First, likelihoods of primary and secondary failures are being evaluated:"
        )
    nx_graph_mod, importance_all_lines, selected_edges = (
        extend_transmission_capacity.increase_capacity_most_important_lines(
            network,
            nx_graph,
            nn_links_extended,
            delta_num_parallel,
            co2l,
            casc_dict=casc_dict,
            extension_type="most_impactful_primary",
        )
    )

    # Check if the lines have been correctly extend
    assert len(selected_edges) == nn_links_extended
    assert all(
        [
            abs(
                (
                    nx_graph_mod.edges[xx]["num_parallel"]
                    - nx_graph.edges[xx]["num_parallel"]
                )
                - delta_num_parallel
            )
            < 1e-8
            for xx in selected_edges
        ]
    )
    assert all(
        [
            abs(
                nx_graph_mod.edges[xx]["num_parallel"]
                - nx_graph.edges[xx]["num_parallel"]
            )
            < 1e-8
            for xx in nx_graph.edges
            if xx not in selected_edges
        ]
    )
    assert len(selected_edges) == nn_links_extended
    assert all(
        [
            abs(
                (
                    nx_graph_mod.edges[xx]["s_nom"] / nx_graph.edges[xx]["s_nom"]
                    - nx_graph_mod.edges[xx]["num_parallel"]
                    / nx_graph.edges[xx]["num_parallel"]
                )
            )
            < 1e-8
            for xx in selected_edges
        ]
    )
    assert all(
        [
            abs(nx_graph_mod.edges[xx]["s_nom"] - nx_graph.edges[xx]["s_nom"])
            < 1e-8
            for xx in nx_graph.edges
            if xx not in selected_edges
        ]
    )

    # Overwrite graph with modified graph
    I_m, B_d, num_parallels, line_limits = data_handling.get_matrices_from_nx_graph(
        nx_graph_mod
    )
    nx_original = nx_graph
    nx_graph = nx_graph_mod

    # Calculate possible N-1 and N-2 failures (using non-bridges)
    bridge_idxs = data_handling.nx_edges_to_matrix_indices(
        nx.bridges(nx_graph), nx_graph
    )
    n_2_failures = cascade_simulation.calc_possible_double_line_failures(
        num_parallels, ignored_idxs=bridge_idxs, use_sclopf=use_sclopf
    )
    n_1_failures = cascade_simulation.calc_possible_single_line_failures(
        num_parallels, ignored_idxs=bridge_idxs
    )

if check_n1_security:
    ### Check N-1 stability ###
    print("\n#### N-1 failures: Co2 level", co2l, " | #Nodes:", n_nodes, " ####")
    for snapshot in tqdm(network.snapshots):

        key_now = snapshot.strftime("%Y-%m-%d %H:00")

        if (
            stop_timestamp_str is not None
            and dt.strptime(stop_timestamp_str, "%Y-%m-%d %H:00") > snapshot
        ):
            break

        P_0 = data_handling.get_effective_injections(network, snapshot, nx_graph)

        for initial_failure in n_1_failures:

            failing_links, system_split = cascade_simulation.simulate_cascade(
                I_m,
                B_d,
                P_0,
                line_limits,
                num_parallels,
                initial_failure,
                max_cascade_length=1,
                use_sclopf=use_sclopf,
            )
            if len(failing_links) > 1:
                raise (RuntimeError("PyPSA networks are not N-1 stable!"))

# ### Simulation of N-2 failures ###
# print("\n#### N-2 failures: Co2 level", co2l, " | #Nodes:", n_nodes, " ####")
# splitting_cascades = dict()
# for snapshot in tqdm(network.snapshots):
#     key_now = snapshot.strftime("%Y-%m-%d %H:00")

#     if stop_timestamp_str is not None:
#         print(f"running {key_now}, stop at {stop_timestamp_str}")
#     # splitting_cascades[snapshot.strftime('%Y-%m-%d %H:00')] = {}

#     P_0 = data_handling.get_effective_injections(network, snapshot, nx_graph)

#     res_dict = dict()
#     for initial_failure in tqdm(n_2_failures, leave=False):

#         failing_links, system_split = cascade_simulation.simulate_cascade(
#             I_m,
#             B_d,
#             P_0,
#             line_limits,
#             num_parallels,
#             initial_failure,
#             use_sclopf=use_sclopf,
#         )

#         if save_all_cascades:
#             # splitting_cascades[snapshot.strftime('%Y-%m-%d %H:00')][tuple(initial_failure)] = failing_links
#             res_dict[tuple(initial_failure)] = failing_links, system_split

#         elif system_split:
#             # splitting_cascades[snapshot.strftime('%Y-%m-%d %H:00')][tuple(initial_failure)] = failing_links
#             res_dict[tuple(initial_failure)] = failing_links

#     splitting_cascades[key_now] = res_dict

#     if (
#         stop_timestamp_str is not None
#         and dt.strptime(stop_timestamp_str, "%Y-%m-%d %H:00") <= snapshot
#     ):
#         break

# if save_all_cascades:
#     fpath_out += "_allcascades"

# if line_mitigation_dict is not None:
#     fpath_out += f"_lineextension_nnlines{nn_links_extended}_deltanumpara{delta_num_parallel:.4g}"

# if stop_timestamp_str is not None:
#     fpath_out += f"_stopped_{stop_timestamp_str}"

# with gzip.open(fpath_out + ".pklz", "wb") as handle:
#     if line_mitigation_dict is None:
#         pickle.dump(splitting_cascades, handle, protocol=pickle.HIGHEST_PROTOCOL)
#     else:
#         pickle.dump(
#             (importance_all_lines, selected_edges, splitting_cascades),
#             handle,
#             protocol=pickle.HIGHEST_PROTOCOL,
#         )

/home/mtitz/.conda/envs/system_split/lib/python3.12/site-packages/pypsa/components.py:323: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attrs.loc[bool_b, "default"] = attrs.loc[bool_b].isin({True, "True"})
/home/mtitz/.conda/envs/system_split/lib/python3.12/site-packages/pypsa/components.py:323: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attrs.loc[bool_b, "default"] = attrs.loc[bool_b].isin({True, "True"})
/home/mtitz/.conda/envs/system_split/lib/python3.12/site-packages/pypsa/components.py:323: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dt

INFO:pypsa.io:Imported network sclopf-elec_s_400_ec_lv1.0_Co2L0.1-2920SEG.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units
/home/mtitz/power-system-split/scripts/../utils/data_handling.py:23: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  snet = pypsa_network.sub_networks["obj"][snet_index]



## Running for line extension: Loading previously found cascade dictionary!
 First, likelihoods of primary and secondary failures are being evaluated:


/home/mtitz/power-system-split/scripts/../utils/extend_transmission_capacity.py:97: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  split_properties_df.lost_load_rocof_share_weighted = (



#### N-1 failures: Co2 level 0.1  | #Nodes: 400  ####


  0%|          | 0/2920 [00:00<?, ?it/s]


RuntimeError: PyPSA networks are not N-1 stable!

In [4]:
import numpy as np
from utils.cascade_simulation import remove_line_from_Bd
def compare_simulate_cascade(
    II_in,
    B_d_in,
    P0,
    line_limits_in,
    num_parallel_in,
    failure_lines,
    epsilon=1e-4,
    max_cascade_length=np.inf,
    use_sclopf: bool = True,
    initial_remove_all: bool = False,
):
    II = II_in.copy()
    B_d = B_d_in.copy()
    num_parallel_ls = num_parallel_in.copy()
    line_limits = line_limits_in.copy()

    # Remove initial failures
    for del_idx in failure_lines:
        remove_line_from_Bd(
            B_d,
            num_parallel_ls,
            line_limits,
            del_idx,
            use_sclopf=use_sclopf,
            remove_all_circuits=initial_remove_all,
        )
    return B_d, num_parallel_ls, line_limits

In [9]:
I_m, B_d, num_parallels, line_limits = data_handling.get_matrices_from_nx_graph(
        nx_graph
    )
P0 = P_0
a = compare_simulate_cascade(
    I_m,
    B_d,
    P0,
    line_limits,
    num_parallels,
    [144])
I_m_mod, B_d_mod, num_parallels_mod, line_limits_mod = data_handling.get_matrices_from_nx_graph(
        nx_graph_mod
    )
P0_mod = P0
b = compare_simulate_cascade(
    I_m_mod,
    B_d_mod,
    P0_mod,
    line_limits_mod,
    num_parallels_mod,
    [144])

NameError: name 'P_0' is not defined

In [29]:
for i,(edge, edge_mod) in enumerate(zip(nx_original.edges(data=True), nx_graph_mod.edges(data=True))):
    if edge!=edge_mod:
        print(edge, edge_mod)

('CH1 0', 'IT1 4', {'weight': 7142.082335431027, 'orientation': ('CH1 0', 'IT1 4'), 'line_index': ['75_outage'], 's_nom': 3396.2052234810544, 'num_parallel': 2.0}) ('CH1 0', 'IT1 4', {'weight': 7142.082335431027, 'orientation': ('CH1 0', 'IT1 4'), 'line_index': ['75_outage'], 's_nom': 5094.307835221582, 'num_parallel': 3.0})


In [8]:
import os
os.listdir(full_path_to_file)

FileNotFoundError: [Errno 2] No such file or directory: '/media/data/system_split//data/European_networks_lopf/elec_s_400_ec_lv1.0_Co2L0.1-3H.nc'

In [ ]:


B_d = sparse.spdiags(
        np.array([attribs["weight"] for u, v, attribs in nx_graph.edges(data=True)]),
        0,
        nx_graph.number_of_edges(),
        nx_graph.number_of_edges(),
    ).asformat("csr")

In [31]:
len([attribs["weight"] for u, v, attribs in nx_graph.edges(data=True)])

547